# **1. Perkenalan Dataset**


Tahap pertama, Anda harus mencari dan menggunakan dataset dengan ketentuan sebagai berikut:

1. **Sumber Dataset**:  
   Dataset dapat diperoleh dari berbagai sumber, seperti public repositories (*Kaggle*, *UCI ML Repository*, *Open Data*) atau data primer yang Anda kumpulkan sendiri.


# **2. Import Library**

Pada tahap ini, Anda perlu mengimpor beberapa pustaka (library) Python yang dibutuhkan untuk analisis data dan pembangunan model machine learning atau deep learning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')
print("Library berhasil diimpor.")

# **3. Memuat Dataset**

Pada tahap ini, Anda perlu memuat dataset ke dalam notebook. Jika dataset dalam format CSV, Anda bisa menggunakan pustaka pandas untuk membacanya. Pastikan untuk mengecek beberapa baris awal dataset untuk memahami strukturnya dan memastikan data telah dimuat dengan benar.

Jika dataset berada di Google Drive, pastikan Anda menghubungkan Google Drive ke Colab terlebih dahulu. Setelah dataset berhasil dimuat, langkah berikutnya adalah memeriksa kesesuaian data dan siap untuk dianalisis lebih lanjut.

Jika dataset berupa unstructured data, silakan sesuaikan dengan format seperti kelas Machine Learning Pengembangan atau Machine Learning Terapan

In [ ]:
# Dataset: Breast Cancer Wisconsin (klasifikasi biner: malignant/benign)
# Sudah disiapkan dalam format CSV (breast_cancer_raw.csv) satu folder di atas notebook ini
df = pd.read_csv('../breast_cancer_raw.csv')

print("Ukuran dataset:", df.shape)
df.head()

# **4. Exploratory Data Analysis (EDA)**

Pada tahap ini, Anda akan melakukan **Exploratory Data Analysis (EDA)** untuk memahami karakteristik dataset.

Tujuan dari EDA adalah untuk memperoleh wawasan awal yang mendalam mengenai data dan menentukan langkah selanjutnya dalam analisis atau pemodelan.

In [ ]:
# 4.1 Informasi umum dataset
df.info()

### 4.2 Statistik deskriptif

In [ ]:
df.describe()

### 4.3 Distribusi target (label)

In [ ]:
target_counts = df['target'].value_counts()
print(target_counts)

plt.figure(figsize=(5,4))
sns.countplot(x='target', data=df)
plt.title('Distribusi Kelas Target (0=malignant, 1=benign)')
plt.xlabel('Target')
plt.ylabel('Jumlah')
plt.show()

### 4.4 Pengecekan missing value & duplikat

In [ ]:
print("Jumlah missing value per kolom:")
print(df.isna().sum()[df.isna().sum() > 0])
print("\nTotal baris duplikat:", df.duplicated().sum())

### 4.5 Korelasi antar fitur (heatmap ringkas)

In [ ]:
plt.figure(figsize=(12, 10))
corr = df.corr(numeric_only=True)
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False)
plt.title('Heatmap Korelasi Antar Fitur')
plt.show()

### 4.6 Deteksi outlier (boxplot contoh beberapa fitur)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['mean radius', 'mean area', 'mean smoothness']):
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

# **5. Data Preprocessing**

Pada tahap ini, data preprocessing adalah langkah penting untuk memastikan kualitas data sebelum digunakan dalam model machine learning.

Jika Anda menggunakan data teks, data mentah sering kali mengandung nilai kosong, duplikasi, atau rentang nilai yang tidak konsisten, yang dapat memengaruhi kinerja model. Oleh karena itu, proses ini bertujuan untuk membersihkan dan mempersiapkan data agar analisis berjalan optimal.

Berikut adalah tahapan-tahapan yang bisa dilakukan, tetapi **tidak terbatas** pada:
1. Menghapus atau Menangani Data Kosong (Missing Values)
2. Menghapus Data Duplikat
3. Normalisasi atau Standarisasi Fitur
4. Deteksi dan Penanganan Outlier
5. Encoding Data Kategorikal
6. Binning (Pengelompokan Data)

Cukup sesuaikan dengan karakteristik data yang kamu gunakan yah. Khususnya ketika kami menggunakan data tidak terstruktur.

In [ ]:
# 5.1 Menangani missing value -> imputasi dengan median (robust terhadap outlier)
df_clean = df.copy()
num_cols = df_clean.columns.drop('target')

for col in num_cols:
    if df_clean[col].isna().sum() > 0:
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val)

print("Sisa missing value setelah imputasi:", df_clean.isna().sum().sum())

### 5.2 Menghapus data duplikat

In [ ]:
before = df_clean.shape[0]
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
after = df_clean.shape[0]
print(f"Jumlah baris sebelum: {before}, setelah hapus duplikat: {after}")

### 5.3 Deteksi & penanganan outlier (metode IQR, capping/winsorizing)

In [ ]:
def cap_outliers_iqr(data, columns):
    data = data.copy()
    for col in columns:
        Q1 = data[col].quantile(0.25)
        Q3 = data[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        data[col] = data[col].clip(lower=lower, upper=upper)
    return data

df_clean = cap_outliers_iqr(df_clean, num_cols)
print("Outlier pada fitur numerik telah di-cap menggunakan metode IQR.")

### 5.4 Pemisahan fitur (X) dan target (y), lalu split train-test

In [ ]:
X = df_clean.drop(columns=['target'])
y = df_clean['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)

### 5.5 Normalisasi/Standarisasi fitur (StandardScaler)

In [ ]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

X_train_scaled.head()

### 5.6 Menyimpan hasil preprocessing (siap dilatih)

In [ ]:
import os
os.makedirs('breast_cancer_preprocessing', exist_ok=True)

train_final = X_train_scaled.copy()
train_final['target'] = y_train.values
test_final = X_test_scaled.copy()
test_final['target'] = y_test.values

train_final.to_csv('breast_cancer_preprocessing/train.csv', index=False)
test_final.to_csv('breast_cancer_preprocessing/test.csv', index=False)

print("Dataset hasil preprocessing tersimpan di folder 'breast_cancer_preprocessing/'")
print("train.csv:", train_final.shape, "| test.csv:", test_final.shape)

# **6. Kesimpulan**

Notebook ini telah melalui seluruh tahapan eksperimen: perkenalan dataset, import library, memuat data, EDA (distribusi target, missing value, duplikat, korelasi, outlier), hingga data preprocessing (imputasi, deduplikasi, penanganan outlier, split data, dan standarisasi fitur). Hasil akhir berupa `train.csv` dan `test.csv` yang siap digunakan untuk pelatihan model pada tahap berikutnya (`automate_Nama-siswa.py` dan `modelling.py`).